<a href="https://colab.research.google.com/github/OlhaZahrebelna/certflow-rag-assistant/blob/main/src/retrieval/03_reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.8 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import faiss

from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder

In [3]:
repo_path = Path("/content/certflow-rag-assistant")

if not repo_path.exists():
    !git clone https://github.com/OlhaZahrebelna/certflow-rag-assistant.git
else:
    print("Repository already exists.")

Cloning into 'certflow-rag-assistant'...
remote: Enumerating objects: 194, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 194 (delta 56), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (194/194), 310.95 KiB | 2.70 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [4]:
%cd /content/certflow-rag-assistant

/content/certflow-rag-assistant


In [5]:
chunks_path = Path("data/raw/processed/chunks.json")

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [chunk["content"] for chunk in chunks]

print(f"Loaded chunks: {len(chunks)}")

Loaded chunks: 81


In [6]:
embedding_model = SentenceTransformer(
    "multi-qa-MiniLM-L6-cos-v1"
)

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"Vectors indexed: {index.ntotal}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Vectors indexed: 81


In [7]:
def retrieve(query, k=10):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = index.search(
        query_embedding,
        k=k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "dense_score": float(scores[0][rank - 1]),
            "chunk_id": chunk["chunk_id"],
            "document": chunk["metadata"]["title"],
            "section": chunk["metadata"]["section"],
            "content": chunk["content"],
        })

    return results

In [8]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [9]:
def rerank(query, candidates, top_k=5):
    pairs = [
        (query, candidate["content"])
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for candidate, score in zip(candidates, scores):
        result = candidate.copy()
        result["reranker_score"] = float(score)

        reranked.append(result)

    reranked = sorted(
        reranked,
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    for rank, result in enumerate(reranked, start=1):
        result["rerank"] = rank

    return reranked[:top_k]

In [10]:
def retrieve_and_rerank(
    query,
    retrieve_k=10,
    final_k=5
):
    candidates = retrieve(
        query,
        k=retrieve_k
    )

    reranked = rerank(
        query,
        candidates,
        top_k=final_k
    )

    return reranked

In [11]:
query = "What evidence is required before an account can be certified?"

results = retrieve_and_rerank(
    query,
    retrieve_k=10,
    final_k=5
)

for result in results:
    print(
        result["rerank"],
        round(result["reranker_score"], 4),
        result["document"],
        "→",
        result["section"]
    )

1 4.4581 Account Data Certification Overview → 4. Account certification outcomes
2 3.3395 Account Certification Frequently Asked Questions → General
3 3.131 End-to-End Account Certification Workflow → 9. Certification decision
4 2.4957 Account Data Certification Overview → 4. Account certification outcomes
5 2.431 Roles and Responsibilities → 1. Requester


In [12]:
def evaluate_reranker_hit_at_k(
    evaluation_dataset,
    k=5,
    retrieve_k=10
):
    hits = 0
    details = []

    for item in evaluation_dataset:
        results = retrieve_and_rerank(
            item["query"],
            retrieve_k=retrieve_k,
            final_k=k
        )

        is_hit = False
        hit_rank = None

        for result in results:
            for expected in item["expected_sections"]:

                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    is_hit = True
                    hit_rank = result["rerank"]
                    break

            if is_hit:
                break

        if is_hit:
            hits += 1

        details.append({
            "query": item["query"],
            "hit": is_hit,
            "rank": hit_rank
        })

    return hits / len(evaluation_dataset), details

In [16]:
def evaluate_reranker_mrr(
    evaluation_dataset,
    k=5,
    retrieve_k=10
):
    reciprocal_ranks = []

    for item in evaluation_dataset:
        results = retrieve_and_rerank(
            item["query"],
            retrieve_k=retrieve_k,
            final_k=k
        )

        rr = 0

        for result in results:
            for expected in item["expected_sections"]:

                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    rr = 1 / result["rerank"]
                    break

            if rr > 0:
                break

        reciprocal_ranks.append(rr)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [14]:
evaluation_dataset = [
    {
        "query": "What evidence is required before an account can be certified?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "6. Evidence recording"
            },
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "4. Gather evidence"
            }
        ]
    },
    {
        "query": "How should potential duplicate accounts be handled?",
        "expected_sections": [
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "6. Duplicate screening"
            },
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "7. Duplicate outcomes"
            }
        ]
    },
    {
        "query": "When should a certification case be escalated?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "6. Escalation levels"
            }
        ]
    },
    {
        "query": "Who is responsible for performing the quality assurance review?",
        "expected_sections": [
            {
                "document": "Roles and Responsibilities",
                "section": "4. Quality Assurance Reviewer"
            }
        ]
    },
    {
        "query": "What validation rules apply to account fields?",
        "expected_sections": [
            {
                "document": "Account Fields and Validation Rules",
                "section": "2. Core field catalog"
            }
        ]
    },
    {
        "query": "What should an analyst do when two sources contain conflicting information?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "5. Conflicting sources"
            }
        ]
    },
    {
        "query": "Which sources should be preferred when verifying account data?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "2. Primary sources"
            },
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "3. Secondary sources"
            }
        ]
    },
    {
        "query": "What should be recorded when a certification change is made?",
        "expected_sections": [
            {
                "document": "Request Types and Change Management",
                "section": "4. Required audit record"
            },
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "6. Record the change"
            }
        ]
    },
    {
        "query": "What information is required when submitting a new certification request?",
        "expected_sections": [
            {
                "document": "Request Types and Change Management",
                "section": "2. Intake requirements"
            },
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "1. Intake"
            }
        ]
    },
    {
        "query": "How do we verify that we are working with the correct company entity?",
        "expected_sections": [
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "2. Identify the correct entity"
            }
        ]
    },
    {
        "query": "What checks are needed before setting an account to Verified?",
        "expected_sections": [
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "9. Certification decision"
            }
        ]
    },
    {
        "query": "When is quality assurance mandatory?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "2. Mandatory QA triggers"
            }
        ]
    },
    {
        "query": "What should be checked during a QA review?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "3. QA checklist"
            }
        ]
    },
    {
        "query": "What happens when an account does not pass quality review?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "4. QA outcomes"
            }
        ]
    },
    {
        "query": "How should missing address information be handled?",
        "expected_sections": [
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "8. Missing address information"
            }
        ]
    },
    {
        "query": "How should an account address be normalized?",
        "expected_sections": [
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "5. Normalization"
            }
        ]
    },
    {
        "query": "What rules apply when the legal name of an account is updated?",
        "expected_sections": [
            {
                "document": "Account Fields and Validation Rules",
                "section": "3. Legal Name"
            }
        ]
    },
    {
        "query": "How should website and primary domain values be validated?",
        "expected_sections": [
            {
                "document": "Account Fields and Validation Rules",
                "section": "5. Website and Primary Domain"
            }
        ]
    },
    {
        "query": "What should happen when a relevant source is unavailable?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "8. Unavailable sources"
            }
        ]
    },
    {
        "query": "When does a certified account need to be reviewed again?",
        "expected_sections": [
            {
                "document": "Account Data Certification Overview",
                "section": "6. Certification validity"
            }
        ]
    }
]

In [17]:
reranker_hit_rate, reranker_details = (
    evaluate_reranker_hit_at_k(
        evaluation_dataset,
        k=5,
        retrieve_k=10
    )
)

reranker_mrr = evaluate_reranker_mrr(
    evaluation_dataset,
    k=5,
    retrieve_k=10
)

print(f"Reranker Hit@5: {reranker_hit_rate:.2f}")
print(f"Reranker MRR@5: {reranker_mrr:.3f}")

Reranker Hit@5: 0.75
Reranker MRR@5: 0.571


## Reranking Result

A CrossEncoder reranker (`cross-encoder/ms-marco-MiniLM-L6-v2`) was evaluated on top of the dense FAISS retriever.

| Retrieval Strategy | Hit@5 | MRR@5 |
|---|---:|---:|
| Dense FAISS baseline | 0.85 | 0.618 |
| Dense FAISS + CrossEncoder reranker | 0.75 | 0.571 |

The reranker reduced both retrieval coverage and ranking quality.

For several queries, the CrossEncoder promoted broadly related passages over the most specific policy sections. Therefore, the reranker was not selected for the final retrieval pipeline.

The dense `multi-qa-MiniLM-L6-cos-v1` + FAISS retriever remains the best-performing retrieval strategy.